# laboratorio 4:

## carga de archivo:
se cargaron los archivos climateChange.txt y renwableEnergy.txt para iniciar su limpieza. tambien se vio cuantas palabras tiene el texto


In [147]:
import os
from pathlib import Path

climateChange = Path("texts/climateChange.txt").read_text(encoding="utf-8")
renewableEnergy = Path("texts/renewableEnergy.txt").read_text(encoding="utf-8")

climateChangeWords = climateChange.split()
renewableEnergyWords = renewableEnergy.split()

print(f"words count in climateChange.txt: {len(climateChangeWords)}")
print(f"words count in renewableEnergy.txt: {len(renewableEnergyWords)}")


words count in climateChange.txt: 368
words count in renewableEnergy.txt: 370


## limpieza de texto:

el texto se limpiara de la siguiente manera:
1. las mayusculas seran reemplazados por minusculas
2. los signos de puntutacion se eliminaran
3. los numeros seran reemplazados por \<num\>



In [148]:
import re 

def cleaner(text):
    #text = re.sub(r"[^а-яё\sА-ЯЁ\d]", " ", text )
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r"\d+", "<num>", text)
    text = text.lower()
    return text

climateChangeCleaned = cleaner(climateChange)
renewableEnergyCleaned = cleaner(renewableEnergy)
#print(climateChangeCleaned)

climateChangeCleanedTokens = climateChangeCleaned.split()
renewableEnergyCleanedTokens = renewableEnergyCleaned.split()



print(f"cantidad de palabras en climateChange.txt: {len(climateChangeCleanedTokens)}")
print(f"cantidad de palabras en renewableEnergy.txt: {len(renewableEnergyCleanedTokens)}")



cantidad de palabras en climateChange.txt: 372
cantidad de palabras en renewableEnergy.txt: 370



### lematizacion:
usando la libreria pymorphy3 se formaran los lemas

In [149]:
import pymorphy3

analyzer = pymorphy3.MorphAnalyzer()

climateChangeLema = [analyzer.parse(palabra)[0].normal_form for palabra in climateChangeCleanedTokens]
renewableEnergyLema  = [analyzer.parse(palabra)[0].normal_form for palabra in renewableEnergyCleanedTokens]

display(climateChangeLema)


['изменение',
 'климат',
 'и',
 'будущее',
 'общество',
 'изменение',
 'климат',
 'становиться',
 'один',
 'из',
 'главный',
 'проблема',
 'современный',
 'общество',
 'под',
 'это',
 'понятие',
 'понимать',
 'длительный',
 'изменение',
 'средний',
 'температура',
 'количество',
 'осадки',
 'и',
 'другой',
 'характеристика',
 'климатический',
 'система',
 'сегодня',
 'деятельность',
 'человек',
 'усиливать',
 'парниковый',
 'эффект',
 'при',
 'сжигание',
 'уголь',
 'нефть',
 'и',
 'газ',
 'в',
 'атмосфера',
 'поступать',
 'углекислый',
 'газ',
 'сельский',
 'хозяйство',
 'и',
 'добыча',
 'топливо',
 'также',
 'сопровождаться',
 'выброс',
 'метан',
 'этот',
 'газа',
 'задерживать',
 'тепло',
 'и',
 'постепенно',
 'изменять',
 'условие',
 'жизнь',
 'на',
 'планета',
 'последствие',
 'потепление',
 'проявляться',
 'по',
 'разный',
 'в',
 'один',
 'регион',
 'частый',
 'возникать',
 'продолжительный',
 'засуха',
 'в',
 'другой',
 'увеличиваться',
 'риск',
 'сильный',
 'осадки',
 'и',
 'нав

### filtrado de palabras vacias


In [150]:
import nltk

tribialTokens = nltk.corpus.stopwords.words("russian")

def tokenFilter(filtr, tokens):
    temp = []
    for lema in tokens:
        flag = True
        for word in filtr:
            if lema == word:
                flag = False
                break
        if flag:
            #print(lema)
            temp.append(lema)
    return temp

climateChangeLema = tokenFilter(tribialTokens, climateChangeLema)
renewableEnergyLema = tokenFilter(tribialTokens, renewableEnergyLema)





### vocabulario comun

In [151]:
vocabulary = sorted(
    set(climateChangeLema) | set(renewableEnergyLema)
)

print(vocabulary)

['<num>', 'агентство', 'адаптироваться', 'аккумулятор', 'атмосфера', 'безопасный', 'бизнес', 'благоприятный', 'большой', 'борьба', 'будущее', 'будущий', 'важно', 'важный', 'ветер', 'ветровый', 'вид', 'влага', 'влиять', 'вместе', 'вода', 'водохранилище', 'воздействие', 'воздушный', 'возможность', 'возможный', 'возникать', 'возобновлять', 'волна', 'восстанавливаться', 'вращать', 'время', 'всё', 'выбор', 'выброс', 'вызывать', 'вынудить', 'вырабатывать', 'выработать', 'выработка', 'выращивание', 'газ', 'газа', 'генератор', 'генерация', 'гидроэлектростанция', 'главный', 'год', 'город', 'государство', 'готовиться', 'гражданин', 'группа', 'давление', 'данные', 'дать', 'двигаться', 'движение', 'дерево', 'десять', 'деятельность', 'длительный', 'добыча', 'должный', 'доля', 'дом', 'дополнительный', 'достигнуть', 'доступность', 'доступный', 'жара', 'животное', 'жизнь', 'жилой', 'зависеть', 'зависимость', 'задерживать', 'замена', 'заранее', 'засуха', 'затопление', 'затрагивать', 'защита', 'здание',

### tabla bow


In [152]:
from collections import Counter
import numpy as np
import pandas as pd

conteo_clima = Counter(climateChangeLema)
conteo_energia = Counter(renewableEnergyLema)

bow = np.array([
    [conteo_clima[lema] for lema in vocabulary],
    [conteo_energia[lema] for lema in vocabulary]
])

print(f"length of vocabulary: {len(vocabulary)}")
print(f"length of BOW: {len(bow[0])}")



tabla_bow = pd.DataFrame(
    bow,
    index=["climate change", "renewable energy"],
    columns=vocabulary
)

display(tabla_bow)

print(bow.sum(axis=1))
print(len(climateChangeLema), len(renewableEnergyLema))

length of vocabulary: 379
length of BOW: 379


,<num>,агентство,адаптироваться,аккумулятор,атмосфера,безопасный,бизнес,благоприятный,большой,борьба,...,электрический,электричество,электростанция,электроэнергия,энергетика,энергетический,энергия,энергоэффективность,это,эффект
climate change,8,0,1,0,1,1,1,0,1,0,...,1,0,0,0,1,2,0,1,2,1
renewable energy,6,1,0,1,0,0,1,1,0,1,...,3,3,1,1,4,3,6,1,1,0


[302 302]
302 302


## primer enfoque:

### calculo de TF

In [153]:
totals = bow.sum(axis=1, keepdims=True)
tf = bow / totals

display(tf)

array([[0.02649007, 0.        , 0.00331126, 0.        , 0.00331126,
        0.00331126, 0.00331126, 0.        , 0.00331126, 0.        ,
        0.00331126, 0.00331126, 0.00662252, 0.00331126, 0.        ,
        0.00331126, 0.00331126, 0.00662252, 0.        , 0.        ,
        0.00662252, 0.        , 0.        , 0.        , 0.00662252,
        0.        , 0.00331126, 0.        , 0.00331126, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.00662252,
        0.00331126, 0.00331126, 0.        , 0.        , 0.        ,
        0.00331126, 0.01324503, 0.00331126, 0.        , 0.        ,
        0.        , 0.00331126, 0.00993377, 0.00331126, 0.00331126,
        0.00331126, 0.00331126, 0.00331126, 0.00331126, 0.        ,
        0.00331126, 0.        , 0.        , 0.00331126, 0.        ,
        0.00331126, 0.00331126, 0.00331126, 0.00993377, 0.        ,
        0.        , 0.00331126, 0.        , 0.        , 0.00331126,
        0.00993377, 0.00331126, 0.00662252, 0.  

### calculo df

In [154]:
df = (bow > 0).sum(axis=0)
display(df)

array([2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1,
       1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1,
       2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1,
       1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 2, 1, 1, 1,
       1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2, 1, 1, 1,
       1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1,
       1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 2, 1, 1,
       1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       2, 1, 1, 2, 2, 1, 1, 2, 1, 2, 1, 2, 2, 2, 1,

### calculo IDF

In [155]:
N = bow.shape[0]
idf = np.log(N / df)
print(idf)

[0.         0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.         0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.         0.         0.69314718 0.         0.69314718 0.69314718
 0.69314718 0.69314718 0.         0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.         0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.
 0.69314718 0.         0.69314718 0.         0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.         0.69314718 0.69314718
 0.         0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.         0.69314718 0.         0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.69314

### TF-IDF

In [156]:
tf_idf = tf * idf

print(tf_idf)



[[0.         0.         0.00229519 0.         0.00229519 0.00229519
  0.         0.         0.00229519 0.         0.00229519 0.00229519
  0.         0.         0.         0.         0.00229519 0.00459038
  0.         0.         0.         0.         0.         0.
  0.00459038 0.         0.00229519 0.         0.00229519 0.
  0.         0.         0.         0.         0.         0.00229519
  0.00229519 0.         0.         0.         0.00229519 0.
  0.00229519 0.         0.         0.         0.00229519 0.
  0.00229519 0.         0.00229519 0.         0.00229519 0.00229519
  0.         0.00229519 0.         0.         0.00229519 0.
  0.00229519 0.00229519 0.00229519 0.         0.         0.
  0.         0.         0.         0.00229519 0.00688557 0.00229519
  0.         0.         0.         0.         0.00229519 0.
  0.00229519 0.00229519 0.00229519 0.         0.00229519 0.
  0.00229519 0.00229519 0.         0.00229519 0.         0.00229519
  0.         0.         0.         0.       

### palabras con mayor peso en ambos textos

In [157]:
saved_top_5 = {}

for name, row in zip(
    ["climate change", "renewable energy"],
    tf_idf
):
    wordsWithWeight = zip(vocabulary, row)

    top_5 = sorted(
        wordsWithWeight,
        key=lambda elemento: (-elemento[1], elemento[0])
    )[:5]

    

    print(name + ":")
    #print()
    for word, weight in top_5:
        print(f"   {word}: {weight:.6f}")
    
    print()
    saved_top_5[name] = top_5


print(saved_top_5)

climate change:
   жара: 0.006886
   общество: 0.006886
   особенно: 0.006886
   средний: 0.006886
   влага: 0.004590

renewable energy:
   энергия: 0.013771
   возобновлять: 0.011476
   источник: 0.009181
   ветер: 0.006886
   использовать: 0.006886

{'climate change': [('жара', np.float64(0.0068855680188074034)), ('общество', np.float64(0.0068855680188074034)), ('особенно', np.float64(0.0068855680188074034)), ('средний', np.float64(0.0068855680188074034)), ('влага', np.float64(0.004590378679204935))], 'renewable energy': [('энергия', np.float64(0.013771136037614807)), ('возобновлять', np.float64(0.01147594669801234)), ('источник', np.float64(0.00918075735840987)), ('ветер', np.float64(0.0068855680188074034)), ('использовать', np.float64(0.0068855680188074034))]}


## segundo enfoque:
  

In [158]:
mascara = df >= 2

vocabulario_filtrado = np.array(vocabulary)[mascara]
bow_filtrado = bow[:, mascara]

## calculo de TF, IDF y TF-IDF

In [159]:
totales_filtrados = bow_filtrado.sum(axis=1, keepdims=True)

tf_filtrado = np.divide(
    bow_filtrado,
    totales_filtrados,
    out=np.zeros_like(bow_filtrado, dtype=float),
    where=totales_filtrados != 0
)

df_filtrado = (bow_filtrado > 0).sum(axis=0)
idf_filtrado = np.log(N / df_filtrado)

tf_idf_filtrado = tf_filtrado * idf_filtrado



In [160]:
saved_top_5_filtered = {}
for nombre, fila in zip(
    ["climate change", "renewable energy"],
    tf_idf_filtrado
):
    top_5_filtered = sorted(
        zip(vocabulario_filtrado, fila),
        key=lambda elemento: (-elemento[1], elemento[0])
    )[:5]

    print(nombre)

    for palabra, peso in top_5_filtered:
        print(f"   {palabra}: {peso:.6f}")
    print()
    saved_top_5_filtered[nombre] = top_5_filtered
print(saved_top_5_filtered)

climate change
   <num>: 0.000000
   бизнес: 0.000000
   важно: 0.000000
   важный: 0.000000
   ветровый: 0.000000

renewable energy
   <num>: 0.000000
   бизнес: 0.000000
   важно: 0.000000
   важный: 0.000000
   ветровый: 0.000000

{'climate change': [(np.str_('<num>'), np.float64(0.0)), (np.str_('бизнес'), np.float64(0.0)), (np.str_('важно'), np.float64(0.0)), (np.str_('важный'), np.float64(0.0)), (np.str_('ветровый'), np.float64(0.0))], 'renewable energy': [(np.str_('<num>'), np.float64(0.0)), (np.str_('бизнес'), np.float64(0.0)), (np.str_('важно'), np.float64(0.0)), (np.str_('важный'), np.float64(0.0)), (np.str_('ветровый'), np.float64(0.0))]}


## comparativa

In [161]:

names = ["climate change", "renewable energy"]

def comparar_tops(top_without_filtr, top_with_filtro):
    A = {word for word, weight in top_without_filtr}
    B = {word for word, weight in top_with_filtro}

    union = A | B

    return {
        "proporcion_comunes": len(A & B) / len(union) if union else 0.0,
        "proporcion_exclusivas": len(A ^ B) / len(union) if union else 0.0,
        "proporcion_nuevas": len(B - A) / len(B) if B else 0.0
    }

print(top_5)

for name in names:
    metricas = comparar_tops(
        saved_top_5[name],
        saved_top_5_filtered[name]
    )

    print(name)
    for metrica, valor in metricas.items():
        print(f"{metrica}: {valor:.2%}")


[('энергия', np.float64(0.013771136037614807)), ('возобновлять', np.float64(0.01147594669801234)), ('источник', np.float64(0.00918075735840987)), ('ветер', np.float64(0.0068855680188074034)), ('использовать', np.float64(0.0068855680188074034))]
climate change
proporcion_comunes: 0.00%
proporcion_exclusivas: 100.00%
proporcion_nuevas: 100.00%
renewable energy
proporcion_comunes: 0.00%
proporcion_exclusivas: 100.00%
proporcion_nuevas: 100.00%


## Interpretación de los resultados

Sin el filtro de frecuencia documental, TF-IDF destaca palabras que diferencian los textos. En cambio climático aparecen términos como «жара» (calor) y «влага» (humedad), aunque también palabras generales como «особенно» (especialmente). En energías renovables destacan «энергия» (energía), «источник» (fuente) y «ветер» (viento).

Al conservar solo los lemas presentes en ambos documentos, todos tienen df = 2. Como N = 2, el IDF clásico es log(2/2) = 0 y todos los pesos TF-IDF filtrados son cero. Por eso, el top filtrado se elige por orden alfabético y no representa mayor relevancia. «<num>» es un marcador de números, no una palabra temática.

Las métricas comparan los tops antes y después del filtro para cada texto: las palabras comunes y exclusivas se dividen entre el tamaño de la unión; las nuevas, entre el tamaño del top filtrado. En ambos textos se obtiene 0 % de comunes, 100 % de exclusivas y 100 % de nuevas en el top filtrado. Esto indica que las listas no comparten términos, pero no demuestra una mejora: el segundo top está empatado en cero. Las palabras nuevas solo entran al top; no se crean durante el filtrado.

En conclusión, con dos documentos este filtro elimina los términos que los distinguen. El resultado muestra que filtrar palabras poco compartidas puede hacer perder información útil para identificar el tema de cada texto.
